# Diabetic Retinopathy — Manual Evaluation Control Panel

This notebook **does not train models**. Evaluate one completed model run at a time and keep each evaluation's files separate. Supported models:

- `efficientnet` — unweighted EfficientNetB0 baseline
- `efficientnet_weighted` — class-weighted EfficientNetB0
- `densenet` — class-weighted DenseNet121
- `fusion` — initial unweighted EfficientNetB0 + DenseNet121 feature fusion

Use the official `validation` split for model selection. Keep `test` held out until the model choices are settled. Test evaluation requires explicit confirmation and repeated test evaluation on the same model/run is blocked by default. Comparisons must use saved evaluations from the same split and require explicit selections.

Full-data and fixed-manifest training variants are evaluated under separate keys: `efficientnet` vs `efficientnet_undersampled`, `efficientnet_weighted` vs `efficientnet_weighted_undersampled`, `densenet` vs `densenet_undersampled`, and `fusion` vs `fusion_undersampled`. Full-data runs use complete official validation/test splits. Undersampled runs use the fixed selected subsets from the same 70:10:20 manifest (expected validation=1,289 and test=2,578 examples). The evaluation output records the selected-split checksum. Comparing undersampled runs requires the same whole split-manifest checksum, which fixes train, validation and test examples together.


## 1. Environment setup



In [1]:
from configs import tensorflow_setup as _tensorflow_setup
import sys
import numpy as np
import tensorflow as tf
_tensorflow_setup.configure_gpu_memory_growth(tf)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("EfficientNetB0 API:", hasattr(tf.keras.applications, "EfficientNetB0"))
print("DenseNet121 API:", hasattr(tf.keras.applications, "DenseNet121"))

from scripts.evaluate import (
    evaluate_model,
    list_saved_evaluations,
    compare_evaluations,
    MODEL_DISPLAY_NAMES,
)
from scripts.utils import print_run_status

print("Evaluation import successful. No model has been loaded yet.")

Python: 3.10.20
TensorFlow: 2.10.1
NumPy: 1.23.5
GPU devices: []
EfficientNetB0 API: True
DenseNet121 API: True
Evaluation import successful. No model has been loaded yet.


In [2]:
model_names = sorted(MODEL_DISPLAY_NAMES)
for selected_model in model_names:
    print("\n" + "=" * 72)
    print_run_status(selected_model)


Model: densenet
No runs found yet.

Model: densenet_undersampled
run_001: status=completed, epoch=5/5, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\densenet_undersampled\runs\run_001

Model: efficientnet
No runs found yet.

Model: efficientnet_undersampled
run_001: status=completed, epoch=10/10, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_undersampled\runs\run_001

Model: efficientnet_weighted
No runs found yet.

Model: efficientnet_weighted_undersampled
run_001: status=completed, epoch=5/5, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_weighted_undersampled\runs\run_001

Model: fusion
No runs found yet.

Model: fusion_undersampled
run_001: status=completed, epoch=5/5, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\fusion_undersampled\runs\run_001


## 2. Unweighted EfficientNetB0



In [4]:
efficientnet_eval = None
model_choices = ("efficientnet", "efficientnet_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        efficientnet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    efficientnet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Available keys: ('efficientnet', 'efficientnet_undersampled')
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\efficientnet_undersampled\run_001\eval_001
Split: test; examples: 2,578
Accuracy: 0.5974
Macro-F1: 0.4124
Quadratic weighted kappa: 0.44267655190489463
Recall — No DR: 0.6626
Recall — Mild NPDR: 0.2337
Recall — Moderate NPDR: 0.5105
Recall — Severe NPDR: 0.2203
Recall — Proliferative DR (PDR): 0.5690


## 3. Class-weighted EfficientNetB0 


In [5]:
weighted_efficientnet_eval = None
model_choices = ("efficientnet_weighted", "efficientnet_weighted_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        weighted_efficientnet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    weighted_efficientnet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Available keys: ('efficientnet_weighted', 'efficientnet_weighted_undersampled')
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\efficientnet_weighted_undersampled\run_001\eval_001
Split: test; examples: 2,578
Accuracy: 0.5279
Macro-F1: 0.3436
Quadratic weighted kappa: 0.38604366420816283
Recall — No DR: 0.5957
Recall — Mild NPDR: 0.2337
Recall — Moderate NPDR: 0.3316
Recall — Severe NPDR: 0.5085
Recall — Proliferative DR (PDR): 0.5517


## 4. Class-weighted DenseNet121


In [6]:
densenet_eval = None
model_choices = ("densenet", "densenet_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        densenet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    densenet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Available keys: ('densenet', 'densenet_undersampled')
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\densenet_undersampled\run_001\eval_001
Split: test; examples: 2,578
Accuracy: 0.5000
Macro-F1: 0.3160
Quadratic weighted kappa: 0.348849273384758
Recall — No DR: 0.5751
Recall — Mild NPDR: 0.4022
Recall — Moderate NPDR: 0.1789
Recall — Severe NPDR: 0.3051
Recall — Proliferative DR (PDR): 0.6552


## 5. Feature-fusion model



In [7]:
fusion_eval = None
model_choices = ("fusion", "fusion_undersampled")
print("Available keys:", model_choices)
model_name = input("Model key: ").strip()
if model_name not in model_choices:
    raise ValueError(f"Choose one of {model_choices}.")
run_id = input(f"Completed run ID for {model_name} (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        fusion_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    fusion_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Available keys: ('fusion', 'fusion_undersampled')
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\fusion_undersampled\run_001\eval_001
Split: test; examples: 2,578
Accuracy: 0.5935
Macro-F1: 0.3868
Quadratic weighted kappa: 0.42074866099761277
Recall — No DR: 0.6610
Recall — Mild NPDR: 0.2120
Recall — Moderate NPDR: 0.5316
Recall — Severe NPDR: 0.3051
Recall — Proliferative DR (PDR): 0.2931


In [8]:
import pandas as pd

saved_evaluations = list_saved_evaluations()
if saved_evaluations:
    display(pd.DataFrame(saved_evaluations))
else:
    print("No evaluation reports are saved yet.")

,model_name,model_display_name,run_id,evaluation_id,split,training_sampling_mode,max_training_examples_per_class,training_subset_sha256,accuracy,macro_f1,weighted_f1,quadratic_weighted_kappa,output_dir
0,densenet_undersampled,DenseNet121 (class-weighted) (fixed 70:10:20 s...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,0.500000,0.315983,0.555524,0.348849,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...
1,efficientnet_undersampled,EfficientNetB0 (unweighted) (fixed 70:10:20 su...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,0.597362,0.412367,0.632774,0.442677,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...
2,efficientnet_weighted_undersampled,EfficientNetB0 (class-weighted) (fixed 70:10:2...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,0.527929,0.343616,0.578904,0.386044,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...
3,fusion_undersampled,EfficientNetB0 + DenseNet121 fusion (unweighte...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,0.593483,0.386799,0.626179,0.420749,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...


## 6. Compare 

In [9]:
comparison_count = int(input("Number of saved evaluations to compare [4]: ").strip() or 4)
if comparison_count < 2:
    raise ValueError("Enter at least two evaluation reports to compare.")

selected_evaluations = []
for index in range(comparison_count):
    print(f"\nEvaluation {index + 1} of {comparison_count}")
    selected_evaluations.append({
        "model_name": input("Model key (see the full/undersampled keys listed above): ").strip(),
        "run_id": input("Run ID (example run_001): ").strip(),
        "evaluation_id": input("Evaluation ID (example eval_001): ").strip(),
    })

comparison_result = compare_evaluations(selected_evaluations)
display(pd.DataFrame(comparison_result["rows"]))
print("Comparison files:", comparison_result["output_dir"])


Evaluation 1 of 4

Evaluation 2 of 4

Evaluation 3 of 4

Evaluation 4 of 4
Comparison saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\comparisons\comparison_001


,model_name,model,run_id,evaluation_id,split,training_sampling_mode,max_training_examples_per_class,training_subset_sha256,split_manifest_sha256,training_split_manifest_sha256,...,n_examples,accuracy,macro_f1,weighted_f1,quadratic_weighted_kappa,recall_no_dr,recall_mild_npdr,recall_moderate_npdr,recall_severe_npdr,recall_proliferative_dr_pdr
0,densenet_undersampled,DenseNet121 (class-weighted) (fixed 70:10:20 s...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,...,2578,0.500000,0.315983,0.555524,0.348849,0.575119,0.402174,0.178947,0.305085,0.655172
1,efficientnet_undersampled,EfficientNetB0 (unweighted) (fixed 70:10:20 su...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,...,2578,0.597362,0.412367,0.632774,0.442677,0.662625,0.233696,0.510526,0.220339,0.568966
2,efficientnet_weighted_undersampled,EfficientNetB0 (class-weighted) (fixed 70:10:2...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,...,2578,0.527929,0.343616,0.578904,0.386044,0.595677,0.233696,0.331579,0.508475,0.551724
3,fusion_undersampled,EfficientNetB0 + DenseNet121 fusion (unweighte...,run_001,eval_001,test,undersampled,2500,0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf6040...,...,2578,0.593483,0.386799,0.626179,0.420749,0.661044,0.211957,0.531579,0.305085,0.293103


Comparison files: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\comparisons\comparison_001
